# 01 — Single-File Anatomy

Dissect **one** audio file where structure meets signal: acidcat walks the
container (chunks, offsets, field-level breakdown), librosa handles the
waveform/spectrum, and we compute the acoustic metrics from `ACOUSTIC_PROPERTIES.md`.

This is the seed of the interactive explorer. Point `TARGET` at any shiny.

In [ ]:
import acidcat, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
pd.set_option('display.max_colwidth', 60)

PLAYGROUND = Path('~/acidcat-playground').expanduser()
SPECIMENS  = PLAYGROUND / 'specimens'

# change this to any file: a shiny, a sample pack wav, an IQ .wav, whatever
TARGET = SPECIMENS / 'formats' / 'float32.wav'
data = TARGET.read_bytes()
print(f'{TARGET.name}  ({len(data):,} bytes)')

## 1. Structure — what acidcat sees

`walk_file()` returns `(format_label, [chunks])`.

In [ ]:
fmt, chunks, fwarns = acidcat.walk_file(str(TARGET))
print('format:', fmt)
if fwarns: print('file warnings:', fwarns)
df_chunks = pd.DataFrame([
    {'id': c['id'], 'offset': c['offset'], 'size': c['size'],
     'summary': c.get('summary',''), 'warnings': len(c.get('warnings') or [])}
    for c in chunks
])
df_chunks

## 2. Field-level breakdown

Drill into a chunk's parsed fields (offset, length, name, value).

In [ ]:
# first chunk that has a field breakdown
target_chunk = next((c for c in chunks if c.get('fields')), chunks[0])
print(f"chunk '{target_chunk['id']}' @ offset {target_chunk['offset']}")
pd.DataFrame(target_chunk.get('fields', []))

## 3. Byte-structure

acidcat's own ascii view, plus a notebook-native byte-value histogram and a
windowed-entropy trace (flat/low = structured headers, high/near-8 = compressed or random).

In [ ]:
# acidcat's built-in ascii histogram
print(acidcat.viz.byte_histogram(data))


In [ ]:
arr = np.frombuffer(data, dtype=np.uint8)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].bar(np.arange(256), np.bincount(arr, minlength=256), width=1.0)
ax[0].set(title='byte-value distribution', xlabel='byte value', ylabel='count')

# windowed Shannon entropy across the file
W = 96
step = max(1, len(arr) // W)
ent = []
for i in range(0, len(arr) - step + 1, step):
    h = np.bincount(arr[i:i+step], minlength=256) / step
    h = h[h > 0]
    ent.append(float(-(h * np.log2(h)).sum()))
ax[1].plot(np.linspace(0, len(arr), len(ent)), ent)
ax[1].set(title='windowed entropy (bits/byte)', xlabel='offset', ylim=(0, 8.2))
plt.tight_layout(); plt.show()

## 4. Signal domain — waveform + spectrogram

(skips cleanly if the file isn't decodable PCM audio).

In [ ]:
import librosa, librosa.display
try:
    y, sr = librosa.load(str(TARGET), sr=None, mono=False)
    ymono = y if y.ndim == 1 else y.mean(axis=0)
    fig, ax = plt.subplots(2, 1, figsize=(13, 6))
    librosa.display.waveshow(ymono, sr=sr, ax=ax[0]); ax[0].set(title=f'waveform  ({sr} Hz)')
    S = librosa.amplitude_to_db(np.abs(librosa.stft(ymono)), ref=np.max)
    img = librosa.display.specshow(S, sr=sr, x_axis='time', y_axis='log', ax=ax[1])
    ax[1].set(title='spectrogram (dB)'); fig.colorbar(img, ax=ax[1], format='%+2.0f dB')
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'not decodable as audio ({type(e).__name__}: {e}) — structure-only file')

## 5. Acoustic properties

The `ACOUSTIC_PROPERTIES.md` metrics, computed for real.

In [ ]:
try:
    y, sr = librosa.load(str(TARGET), sr=None, mono=True)
    peak = float(np.max(np.abs(y)));  rms = float(np.sqrt(np.mean(y**2)))
    crest = 20*np.log10(peak/rms) if rms>0 else float('nan')
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    cen = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    roll = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
    flat = float(np.mean(librosa.feature.spectral_flatness(y=y)))
    tempo = float(librosa.beat.beat_track(y=y, sr=sr)[0])
    props = {
        'duration_s': round(len(y)/sr, 3), 'sample_rate': sr,
        'peak': round(peak,4), 'rms': round(rms,4), 'crest_factor_dB': round(crest,2),
        'zero_crossing_rate': round(zcr,4), 'spectral_centroid_Hz': round(cen,1),
        'spectral_rolloff_Hz': round(roll,1), 'spectral_flatness': round(flat,4),
        'tempo_bpm': round(tempo,2),
    }
    display(pd.Series(props, name=TARGET.name).to_frame())
except Exception as e:
    print(f'acoustic metrics need decodable audio — skipped ({type(e).__name__})')

---
**Next:** wire `ipywidgets` on `TARGET` (a file picker + a chunk selector that
highlights the bytes) and this becomes the live explorer. Feeds the tweet-thread
write-ups too.